# Phần 4 · Biến đổi Hough và Hậu xử lý tìm hai biên đường

Notebook thực hiện phần việc của Module 4 trong đồ án *Phát hiện biên đường bằng Canny và Hough*:
1. Áp dụng mặt nạ vùng quan tâm (ROI hình thang thích ứng).
2. Phát hiện các đoạn thẳng bằng biến đổi Hough xác suất (Probabilistic Hough - P-Hough).
3. Lọc nhiễu độ dốc (slope filtering) và phân chia hai cụm biên trái / biên phải.
4. Hồi quy tuyến tính có trọng số theo chiều dài đoạn thẳng để xác định hai biên đường.
5. Tạo đa giác mặt đường và đánh giá định lượng đối chiếu với Ground Truth (IoU & MAE).
6. So sánh thực nghiệm Standard Hough (`cv2.HoughLines`) và P-Hough (`cv2.HoughLinesP`).

## Hướng dẫn chạy trên Google Colab
1. Tải thư mục dự án lên Google Drive (ví dụ: `MyDrive/canny-hough-imageprocessing`).
2. Mở notebook này bằng Google Colab.
3. Bỏ dấu comment `#` ở ô mount Drive bên dưới và chỉnh đường dẫn `ROOT` phù hợp.
4. Chạy tuần tự các ô lệnh từ trên xuống dưới.

In [ ]:
import sys
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. Mount Google Drive neu chay tren Colab
# from google.colab import drive
# drive.mount('/content/drive')
# ROOT = Path('/content/drive/MyDrive/canny-hough-imageprocessing')

# Neu chay local hoac repo root hien tai:
ROOT = Path.cwd()
if not (ROOT / 'dataset').is_dir() and (ROOT.parent / 'dataset').is_dir():
    ROOT = ROOT.parent

print('Thu muc du an:', ROOT.resolve())
assert (ROOT / 'outputs' / 'canny' / 'edges').is_dir(), 'Can chay Module Canny truoc de co thu muc outputs/canny/edges'

CANNY_EDGES_DIR = ROOT / 'outputs' / 'canny' / 'edges'
RESIZED_RGB_DIR = ROOT / 'outputs' / 'preprocessed_for_canny' / 'resized'
GT_DIR = ROOT / 'dataset' / 'ground_truth'
OUT_HOUGH = ROOT / 'outputs' / 'hough'
OUT_HOUGH.mkdir(parents=True, exist_ok=True)

GROUPS = ['normal', 'curved', 'adverse']
ALL_IMAGES = [(g, p.stem) for g in GROUPS for p in sorted((CANNY_EDGES_DIR / g).glob('*.png'))]
print(f'Tong cong: {len(ALL_IMAGES)} anh canh Canny san sang xu ly.')

## 1. Cài đặt các hàm xử lý Hough thuần túy

Bao gồm:
- Tính toán 4 đỉnh của ROI hình thang cân qua trục giữa.
- Áp dụng mặt nạ ROI bằng `cv2.fillPoly` và `cv2.bitwise_and`.
- P-Hough (`cv2.HoughLinesP`) và Standard Hough (`cv2.HoughLines`).
- Bộ lọc độ dốc $m = \Delta y / \Delta x$ trong hệ tọa độ pixel ($y$ hướng xuống).
- Hồi quy tuyến tính có trọng số $x = k \cdot y + b$.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class HoughConfig:
    roi_top_width_ratio: float = 0.35      # Day tren ROI (~35% W)
    roi_bottom_width_ratio: float = 0.98   # Day duoi ROI (~98% W)
    roi_top_y_ratio: float = 0.48          # Sat duong chan troi (~48% H)
    roi_bottom_y_ratio: float = 0.98       # Loai bo nap capo (~98% H)

    rho: float = 1.0                       # Do phan giai ban kinh (pixel)
    theta: float = np.pi / 180.0           # Do phan giai goc (rad, 1 do)
    threshold: int = 25                    # So phieu accumulator toi thieu
    min_line_length: float = 25.0          # Chieu dai doan thang toi thieu (pixel)
    max_line_gap: float = 25.0             # Khoang dut doan toi da (pixel)

    min_abs_slope: float = 0.30            # Loai bo vach ngang (|m| < 0.30 ~ 17 do)
    max_abs_slope: float = 4.00            # Loai bo vach qua dung (|m| > 4.00 ~ 76 do)


def get_trapezoid_vertices(height: int, width: int, cfg: HoughConfig) -> np.ndarray:
    center_x = width / 2.0
    top_y = int(height * cfg.roi_top_y_ratio)
    bottom_y = int(height * cfg.roi_bottom_y_ratio)

    half_top_w = (width * cfg.roi_top_width_ratio) / 2.0
    half_bottom_w = (width * cfg.roi_bottom_width_ratio) / 2.0

    top_left = (int(max(0, center_x - half_top_w)), top_y)
    top_right = (int(min(width - 1, center_x + half_top_w)), top_y)
    bottom_right = (int(min(width - 1, center_x + half_bottom_w)), bottom_y)
    bottom_left = (int(max(0, center_x - half_bottom_w)), bottom_y)

    return np.array([[bottom_left, top_left, top_right, bottom_right]], dtype=np.int32)


def apply_roi(edges: np.ndarray, vertices: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    mask = np.zeros_like(edges, dtype=np.uint8)
    cv2.fillPoly(mask, vertices, 255)
    masked_edges = cv2.bitwise_and(edges, mask)
    return masked_edges, mask


def detect_hough_lines_p(roi_edges: np.ndarray, cfg: HoughConfig) -> np.ndarray:
    lines = cv2.HoughLinesP(
        roi_edges,
        rho=cfg.rho,
        theta=cfg.theta,
        threshold=cfg.threshold,
        minLineLength=cfg.min_line_length,
        maxLineGap=cfg.max_line_gap,
    )
    if lines is None:
        return np.empty((0, 4), dtype=np.int32)
    return lines.reshape(-1, 4).astype(np.int32)


def split_left_right_segments(lines: np.ndarray, width: int, cfg: HoughConfig) -> tuple[np.ndarray, np.ndarray]:
    if len(lines) == 0:
        return np.empty((0, 4), dtype=np.int32), np.empty((0, 4), dtype=np.int32)

    left_list, right_list = [], []
    for seg in lines:
        x1, y1, x2, y2 = seg
        dx = x2 - x1
        dy = y2 - y1
        if dx == 0:
            continue
        slope = dy / dx
        abs_slope = abs(slope)
        if not (cfg.min_abs_slope <= abs_slope <= cfg.max_abs_slope):
            continue

        x_mid = (x1 + x2) / 2.0
        if slope < 0 and x_mid < width * 0.65:
            left_list.append(seg)
        elif slope > 0 and x_mid > width * 0.35:
            right_list.append(seg)

    left_arr = np.array(left_list, dtype=np.int32) if left_list else np.empty((0, 4), dtype=np.int32)
    right_arr = np.array(right_list, dtype=np.int32) if right_list else np.empty((0, 4), dtype=np.int32)
    return left_arr, right_arr


def fit_boundary_line(segments: np.ndarray, y_bottom: int, y_top: int) -> tuple[tuple[int, int], tuple[int, int]] | None:
    if len(segments) == 0:
        return None
    dx = segments[:, 2] - segments[:, 0]
    dy = segments[:, 3] - segments[:, 1]
    lengths = np.hypot(dx, dy)
    valid = lengths > 1e-3
    if not np.any(valid):
        return None

    valid_segments = segments[valid]
    weights = lengths[valid]

    x_coords = np.concatenate([valid_segments[:, 0], valid_segments[:, 2]])
    y_coords = np.concatenate([valid_segments[:, 1], valid_segments[:, 3]])
    w = np.concatenate([weights, weights])

    total_w = np.sum(w)
    if total_w <= 0:
        return None

    mean_y = np.sum(w * y_coords) / total_w
    mean_x = np.sum(w * x_coords) / total_w

    var_y = np.sum(w * (y_coords - mean_y) ** 2)
    cov_xy = np.sum(w * (y_coords - mean_y) * (x_coords - mean_x))
    if var_y < 1e-5:
        return None

    k = cov_xy / var_y
    b = mean_x - k * mean_y
    x_bottom = int(round(k * y_bottom + b))
    x_top = int(round(k * y_top + b))
    return ((x_bottom, y_bottom), (x_top, y_top))


def detect_road_boundaries(edges: np.ndarray, cfg: HoughConfig | None = None) -> dict:
    config = cfg or HoughConfig()
    height, width = edges.shape[:2]

    vertices = get_trapezoid_vertices(height, width, config)
    roi_edges, roi_mask = apply_roi(edges, vertices)
    raw_lines = detect_hough_lines_p(roi_edges, config)
    left_segs, right_segs = split_left_right_segments(raw_lines, width, config)

    y_bottom = int(height * config.roi_bottom_y_ratio)
    y_top = int(height * config.roi_top_y_ratio)

    left_line = fit_boundary_line(left_segs, y_bottom, y_top)
    right_line = fit_boundary_line(right_segs, y_bottom, y_top)

    polygon = None
    if left_line is not None and right_line is not None:
        (xl_b, yl_b), (xl_t, yl_t) = left_line
        (xr_b, yr_b), (xr_t, yr_t) = right_line
        polygon = np.array([[xl_b, yl_b], [xl_t, yl_t], [xr_t, yr_t], [xr_b, yr_b]], dtype=np.int32)

    return {
        'left_line': left_line,
        'right_line': right_line,
        'polygon': polygon,
        'raw_lines': raw_lines,
        'left_segments': left_segs,
        'right_segments': right_segs,
        'roi_mask': roi_mask,
    }

## 2. Minh họa trực quan các bước trung gian trên ảnh mẫu

Chạy thử nghiệm trên ảnh `normal_05` để quan sát từng bước:
1. Ảnh cạnh Canny đầu vào.
2. Áp dụng ROI hình thang.
3. Các đoạn thẳng P-Hough thô.
4. Phân loại độ dốc trái (xanh lam) / phải (đỏ).
5. Hai đường biên khớp được và đa giác mặt đường.

In [ ]:
sample_group, sample_name = 'normal', 'normal_05'
edge_img = cv2.imread(str(CANNY_EDGES_DIR / sample_group / f'{sample_name}.png'), cv2.IMREAD_GRAYSCALE)
rgb_img = cv2.cvtColor(cv2.imread(str(RESIZED_RGB_DIR / sample_group / f'{sample_name}.png')), cv2.COLOR_BGR2RGB)

cfg = HoughConfig()
res = detect_road_boundaries(edge_img, cfg)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Subplot 1: Canh Canny
axes[0, 0].imshow(edge_img, cmap='gray')
axes[0, 0].set_title('(a) Canh Canny dau vao', fontsize=11, fontweight='bold')
axes[0, 0].axis('off')

# Subplot 2: ROI Masked edges
vertices = get_trapezoid_vertices(edge_img.shape[0], edge_img.shape[1], cfg)
roi_edges, _ = apply_roi(edge_img, vertices)
axes[0, 1].imshow(roi_edges, cmap='gray')
axes[0, 1].set_title(f'(b) Sau mat na ROI ({len(res["raw_lines"])} doan P-Hough)', fontsize=11, fontweight='bold')
axes[0, 1].axis('off')

# Subplot 3: Doan trai / phai sau loc do doc
seg_canvas = np.zeros((edge_img.shape[0], edge_img.shape[1], 3), dtype=np.uint8)
for x1, y1, x2, y2 in res['left_segments']:
    cv2.line(seg_canvas, (x1, y1), (x2, y2), (255, 100, 100), 2)
for x1, y1, x2, y2 in res['right_segments']:
    cv2.line(seg_canvas, (x1, y1), (x2, y2), (100, 100, 255), 2)
axes[1, 0].imshow(seg_canvas)
axes[1, 0].set_title(f'(c) Loc do doc: Trai ({len(res["left_segments"])}), Phai ({len(res["right_segments"])})', fontsize=11, fontweight='bold')
axes[1, 0].axis('off')

# Subplot 4: Bien hoan chinh
final_canvas = rgb_img.copy()
if res['polygon'] is not None:
    overlay = final_canvas.copy()
    cv2.fillPoly(overlay, [res['polygon']], (0, 200, 0))
    cv2.addWeighted(overlay, 0.3, final_canvas, 0.7, 0, final_canvas)
if res['left_line'] is not None:
    cv2.line(final_canvas, res['left_line'][0], res['left_line'][1], (255, 0, 0), 3)
if res['right_line'] is not None:
    cv2.line(final_canvas, res['right_line'][0], res['right_line'][1], (0, 0, 255), 3)
axes[1, 1].imshow(final_canvas)
axes[1, 1].set_title('(d) Ket qua 2 bien va da giac long duong', fontsize=11, fontweight='bold')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

## 3. So sánh Standard Hough và Probabilistic Hough

- **Standard Hough (`cv2.HoughLines`):** Chiếu từng pixel cạnh lên toàn bộ đường sin trong không gian $(\rho, \theta)$. Đầu ra là các đường thẳng kéo dài vô hạn cắt ngang toàn bộ ảnh, cần xử lý cắt biên phức tạp.
- **Probabilistic Hough (`cv2.HoughLinesP`):** Lấy mẫu ngẫu nhiên các điểm cạnh và liên kết các điểm thẳng hàng thành các đoạn thẳng hữu hạn $[x_1, y_1, x_2, y_2]$, cho phép lọc trực tiếp theo chiều dài đoạn thẳng.

In [ ]:
def compare_hough_methods(edge_img, rgb_img, cfg):
    vertices = get_trapezoid_vertices(edge_img.shape[0], edge_img.shape[1], cfg)
    roi_edges, _ = apply_roi(edge_img, vertices)

    t0 = time.perf_counter()
    std_lines = cv2.HoughLines(roi_edges, rho=cfg.rho, theta=cfg.theta, threshold=cfg.threshold)
    t_std = (time.perf_counter() - t0) * 1000.0

    t0 = time.perf_counter()
    p_lines = cv2.HoughLinesP(roi_edges, rho=cfg.rho, theta=cfg.theta, threshold=cfg.threshold,
                              minLineLength=cfg.min_line_length, maxLineGap=cfg.max_line_gap)
    t_p = (time.perf_counter() - t0) * 1000.0

    print(f'Standard Hough: {0 if std_lines is None else len(std_lines)} duong ({t_std:.2f} ms)')
    print(f'P-Hough:       {0 if p_lines is None else len(p_lines)} doan ({t_p:.2f} ms)')

compare_hough_methods(edge_img, rgb_img, cfg)

## 4. Chạy thực nghiệm trên 50 ảnh và tổng hợp thống kê theo nhóm

Tính toán:
- Tỷ lệ bắt thành công biên Trái, biên Phải và cả Hai biên.
- Chỉ số $IoU$ diện tích vùng đường so với Ground Truth nhị phân.
- Sai số khoảng cách biên $MAE$ (Mean Absolute Error) tính tại 3 mặt cắt ngang $y = 60\%, 75\%, 90\% \\times H$.

In [ ]:
records = []
W, H = 960, 540

for group, name in ALL_IMAGES:
    edge = cv2.imread(str(CANNY_EDGES_DIR / group / f'{name}.png'), cv2.IMREAD_GRAYSCALE)
    gt_raw = cv2.imread(str(GT_DIR / group / f'{name}.png'), cv2.IMREAD_GRAYSCALE)
    gt = cv2.resize(gt_raw, (W, H), interpolation=cv2.INTER_NEAREST) > 0

    t0 = time.perf_counter()
    res = detect_road_boundaries(edge, cfg)
    dt_ms = (time.perf_counter() - t0) * 1000.0

    iou = 0.0
    if res['polygon'] is not None:
        mask = np.zeros((H, W), dtype=np.uint8)
        cv2.fillPoly(mask, [res['polygon']], 1)
        intersection = np.logical_and(mask > 0, gt).sum()
        union = np.logical_or(mask > 0, gt).sum()
        iou = float(intersection / union) if union > 0 else 0.0

    mae = None
    if res['left_line'] is not None and res['right_line'] is not None:
        (xl_b, yl_b), (xl_t, yl_t) = res['left_line']
        (xr_b, yr_b), (xr_t, yr_t) = res['right_line']
        kl = (xl_b - xl_t) / (yl_b - yl_t) if yl_b != yl_t else 0.0
        kr = (xr_b - xr_t) / (yr_b - yr_t) if yr_b != yr_t else 0.0

        errors = []
        for r in [0.60, 0.75, 0.90]:
            yk = int(H * r)
            row_idx = np.where(gt[yk, :])[0]
            if len(row_idx) == 0:
                continue
            xl_gt, xr_gt = row_idx[0], row_idx[-1]
            xl_pred = xl_t + kl * (yk - yl_t)
            xr_pred = xr_t + kr * (yk - yr_t)
            errors.append(abs(xl_pred - xl_gt) + abs(xr_pred - xr_gt))
        if len(errors) > 0:
            mae = float(np.mean(errors) / 2.0)

    records.append({
        'group': group,
        'image': name,
        'raw_lines': len(res['raw_lines']),
        'left_found': int(res['left_line'] is not None),
        'right_found': int(res['right_line'] is not None),
        'both_found': int(res['polygon'] is not None),
        'iou': iou,
        'mae': mae,
        'runtime_ms': dt_ms,
    })

df = pd.DataFrame(records)

summary = []
for g in GROUPS + ['all']:
    sub = df if g == 'all' else df[df['group'] == g]
    summary.append({
        'Nhom': g,
        'So anh': len(sub),
        'P-Hough TB': round(sub['raw_lines'].mean(), 1),
        'Bat Trai (%)': round(sub['left_found'].mean() * 100.0, 1),
        'Bat Phai (%)': round(sub['right_found'].mean() * 100.0, 1),
        'Bat Ca 2 Bien (%)': round(sub['both_found'].mean() * 100.0, 1),
        'IoU TB (ca dat)': round(sub[sub['both_found'] == 1]['iou'].mean(), 4),
        'MAE TB (px)': round(sub[sub['both_found'] == 1]['mae'].mean(), 2),
        'Runtime TB (ms)': round(sub['runtime_ms'].mean(), 2),
    })

df_summary = pd.DataFrame(summary)
display(df_summary)